In [1]:
import os
import sys
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

repo_url = "https://github.com/hoangquan1503/Topic_classification.git" 
repo_name = "Topic_classification"

# if not exist
if not os.path.exists(repo_name):
    print("Cloning repository...")
    !git clone {repo_url}


%cd {repo_name}
print(f"Current: {os.getcwd()}")

!git pull

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

if 'src' in sys.modules:
    del sys.modules['src']
    print("remove duplicate src")

Mounted at /content/drive
Cloning repository...
Cloning into 'Topic_classification'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 15 (delta 2), reused 11 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 275.20 KiB | 25.02 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Topic_classification
Current: /content/Topic_classification
Already up to date.


In [ ]:
from huggingface_hub import login
login(token="hf_") 

In [4]:
from datasets import load_dataset
ds = load_dataset("UniverseTBD/arxiv-abstracts-large")

ds

README.md:   0%|          | 0.00/810 [00:00<?, ?B/s]

arxiv-metadata-oai-snapshot.json: reconstructing file:   0%|          |  0.00B / 3.82GB            

arxiv-metadata-oai-snapshot.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2292057 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})

In [5]:
print(ds['train'][0]['abstract'])
print(ds['train'][0]['categories'])

  A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from quark-antiquark,
gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as
all-orders resummation of initial-state gluon radiation valid at
next-to-next-to-leading logarithmic accuracy. The region of phase space is
specified in which the calculation is most reliable. Good agreement is
demonstrated with data from the Fermilab Tevatron, and predictions are made for
more detailed tests with CDF and DO data. Predictions are shown for
distributions of diphoton pairs produced at the energy of the Large Hadron
Collider (LHC). Distributions of the diphoton pairs from the decay of a Higgs
boson are contrasted with those produced from QCD processes at the LHC, showing
that enhanced sensitivity to the signal can be obtained with judicious
selection of events.

hep-ph


In [6]:
all_categories = ds['train']['categories']
unique_cate = set()

# Split name of categories 
for category in all_categories:
    topic = category.split(' ')[0]
    topic = topic.split('.')[0] #only take first general topic
    unique_cate.add(topic)
    
print(unique_cate)
print(len(unique_cate))
        

{'solv-int', 'hep-ex', 'plasm-ph', 'hep-ph', 'adap-org', 'quant-ph', 'math-ph', 'ao-sci', 'physics', 'hep-lat', 'alg-geom', 'q-fin', 'chem-ph', 'eess', 'hep-th', 'nlin', 'supr-con', 'atom-ph', 'bayes-an', 'q-bio', 'patt-sol', 'cond-mat', 'astro-ph', 'dg-ga', 'comp-gas', 'nucl-ex', 'math', 'q-alg', 'econ', 'gr-qc', 'stat', 'cs', 'cmp-lg', 'acc-phys', 'funct-an', 'nucl-th', 'chao-dyn', 'mtrl-th'}
38


In [7]:
from src.data_preprocessing import processed_ds, preprocess
from datasets import load_from_disk

# Save to Drive
save_path = '/content/drive/MyDrive/Topic_classification/processed_train_dataset' 
if os.path.exists(save_path):
    processed_sample = load_from_disk(save_path)
else:
    processed_sample = processed_ds(ds['train'], preprocess, num_proc=4)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    processed_sample.save_to_disk(save_path)
    print('save done')
    


    

In [8]:
print(processed_sample[:2])

{'id': ['0704.0001', '0704.0002'], 'submitter': ['Pavel Nadolsky', 'Louis Theran'], 'authors': ["C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'Ileana Streinu and Louis Theran'], 'title': ['Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'Sparsity-certifying Graph Decompositions'], 'comments': ['37 pages, 15 figures; published version', 'To appear in Graphs and Combinatorics'], 'journal-ref': ['Phys.Rev.D76:013009,2007', None], 'doi': ['10.1103/PhysRevD.76.013009', None], 'report-no': ['ANL-HEP-PR-07-12', None], 'categories': ['hep-ph', 'math'], 'license': [None, 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/'], 'abstract': ['a fully differential calculation in perturbative quantum chromodynamics ispresented for the production of massive photon pairs at hadron colliders allnexttoleading order perturbative contributions from quarkantiquarkgluonantiquark and gluongluon subprocesses are included as well asallorders resummation of

In [9]:
# Build dict to convert label to num
label_to_id = {label : i for i, label in enumerate(sorted(list(unique_cate)))}
id_to_label = {i : label  for i, label in enumerate(sorted(list(unique_cate)))}

print(label_to_id)



{'acc-phys': 0, 'adap-org': 1, 'alg-geom': 2, 'ao-sci': 3, 'astro-ph': 4, 'atom-ph': 5, 'bayes-an': 6, 'chao-dyn': 7, 'chem-ph': 8, 'cmp-lg': 9, 'comp-gas': 10, 'cond-mat': 11, 'cs': 12, 'dg-ga': 13, 'econ': 14, 'eess': 15, 'funct-an': 16, 'gr-qc': 17, 'hep-ex': 18, 'hep-lat': 19, 'hep-ph': 20, 'hep-th': 21, 'math': 22, 'math-ph': 23, 'mtrl-th': 24, 'nlin': 25, 'nucl-ex': 26, 'nucl-th': 27, 'patt-sol': 28, 'physics': 29, 'plasm-ph': 30, 'q-alg': 31, 'q-bio': 32, 'q-fin': 33, 'quant-ph': 34, 'solv-int': 35, 'stat': 36, 'supr-con': 37}


In [10]:
def convert_label(ds):
    ds['label_id'] = label_to_id[ds['label']]
    return ds

processed_sample = processed_sample.map(convert_label)

Map:   0%|          | 0/2292057 [00:00<?, ? examples/s]

In [11]:
dataset_split = processed_sample.train_test_split(test_size=0.2, seed=42, stratify_by_column='label_id')
X_train = dataset_split['train']['text']
y_train = dataset_split['train']['label_id']
X_test = dataset_split['test']['text']
y_test = dataset_split['test']['label_id']
print(len(X_train))

ValueError: Stratifying by column is only supported for ClassLabel column, and column label_id is Value.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from src.embedding_vector import EmbeddingVectorizer

bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

tfidf =  TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

embeddings = EmbeddingVectorizer()
X_train_embeddings = embeddings.transform(X_train)
X_test_embeddings = embeddings.transform(X_test)

# convert all to numpy for consistency
X_train_bow, X_train_bow = np.array(X_train_bow), np.array(X_test_bow)
X_train_tfidf, X_test_tfidf = np.array(X_train_tfidf), np.array(X_test_tfidf)

print(f'shape of BoW: {X_train_bow.shape}')
print(f'shape of tfidf: {X_train_tfidf.shape}')
print(f'shape of embedding: {X_train_embeddings.shape}')


In [ ]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.cluster import KMeans
import lightgbm 
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
def train_ada(X_train, y_train, X_test, y_test, n_estimators=100, lr=0.1):
    ada = AdaBoostClassifier(n_estimators=n_estimators, learning_rate=lr, random_state=42)
    ada.fit(X_train, y_train)
    y_pred = ada.predict(X_test)
    score = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=sorted(list(unique_cate)), output_dict=True) # set True so that result printed under dict format
    return y_pred, score, report
# BoW result
ada_bow_prediction, ada_bow_score, ada_bow_report = train_ada(X_train_bow, y_train, X_test_bow, y_test)
print(f'AdaBoost BoW Score: {ada_bow_score}\n')
# TFIDF result
ada_tfidf_prediction, ada_tfidf_score, ada_tfidf_report = train_ada(X_train_tfidf, y_train, X_test_tfidf, y_test)
print(f'AdaBoost TFIDF Score: {ada_tfidf_score}\n')
# Embedding result
ada_embed_prediction, ada_embed_score, ada_embed_report = train_ada(X_train_embeddings, y_train, X_test_embeddings, y_test)
print(f'AdaBoost TFIDF Rpeort: {ada_embed_report}')




In [ ]:
# def train_tree(X_train, y_train, X_test, y_test, n_estimators=100, random=42):
#     ranforest = RandomForestClassifier(n_estimators=n_estimators, random_state=random)
#     ranforest.fit(X_train, y_train)
#     y_pred = ranforest.predict(X_test)
#     score = accuracy_score(y_test, y_pred)
#     report = classification_report(y_test, y_pred, target_names=sorted(list(unique_cate)), output_dict=True) # set True so that result printed under dict format
#     return y_pred, score, report

# ranforest_bow_prediction, ranforest_bow_score, ranforest_bow_report = train_ada(X_train_bow, y_train, X_test_bow, y_test)
# print(f'RanForest BoW Score: {ranforest_bow_score}\n')

# ranforest_tfidf_pred, ranforest_tfidf_score, ranforest_tfidf_report = train_tree(X_train_tfidf, y_train, X_test_tfidf, y_test)
# print(f'RanForest TFIDF Score: {ranforest_tfidf_score}\n')

# ranforest_embed_pred, ranforest_embed_score, ranforest_embed_report = train_ada(X_train_embeddings, y_train, X_test_embeddings, y_test)
# print(f'RanForest Embedding Score: {ranforest_embed_score}')

 